## 3.4 Alternative synthetic targets


In [ ]:
new_target_definitions = {
    "Balanced Index Tracking": {
        "MXWD Index": 0.60,
        "LEGATRUU Index": 0.40
    },
    "UCITS Clone": {
        "HFRXGL Index": 0.50,
        "MXWD Index": 0.25,
        "LEGATRUU Index": 0.25
    },
    "Risk Management Target": {
        "HFRXGL Index": 0.40,
        "MXWD Index": 0.40,
        "LEGATRUU Index": 0.20
    }
}

# Check that the weights of each target sum to 1
for target_name, weights in new_target_definitions.items():
    total_weight = sum(weights.values())
    
    if not np.isclose(total_weight, 1.0):
        raise ValueError(
            f"Weights for {target_name} do not sum to 1. Current sum: {total_weight:.4f}"
        )

print("All target weights sum to 1.")

# Display the composition of the new synthetic targets
new_target_composition = pd.DataFrame(new_target_definitions).fillna(0).T

new_target_composition_display = new_target_composition.copy()

for col in new_target_composition_display.columns:
    new_target_composition_display[col] = new_target_composition_display[col].map(
        lambda x: f"{x * 100:.0f}%"
    )

print("New synthetic targets - composition:")
display(new_target_composition_display)

In [ ]:
# Build weekly returns for each new synthetic target
new_targets_returns = pd.DataFrame(index=component_returns.index)

for target_name, weights in new_target_definitions.items():
    target_series = pd.Series(0.0, index=component_returns.index)
    
    for component, weight in weights.items():
        target_series += component_returns[component] * weight
    
    new_targets_returns[target_name] = target_series

# Align with the same dates used for the main synthetic target and futures
new_targets_returns = new_targets_returns.loc[common_dates]

print("New synthetic targets - weekly returns preview:")
display(new_targets_returns.head())



In [ ]:
# Compare main synthetic target with the new synthetic targets
all_synthetic_targets = pd.concat(
    [
        target_returns_aligned.rename("Principal Target"),
        new_targets_returns
    ],
    axis=1
)

all_targets_stats_raw = pd.DataFrame({
    "Ann. Return": all_synthetic_targets.mean() * 52,
    "Ann. Volatility": all_synthetic_targets.std() * np.sqrt(52),
    "Sharpe Ratio": (
        all_synthetic_targets.mean() * 52
    ) / (
        all_synthetic_targets.std() * np.sqrt(52)
    ),
    "Max Drawdown": all_synthetic_targets.apply(max_drawdown),
    "Weekly VaR 95%": all_synthetic_targets.quantile(0.05),
    "Weekly Expected Shortfall 95%": all_synthetic_targets.apply(
        lambda x: x[x <= x.quantile(0.05)].mean()
    ),
    "Skewness": all_synthetic_targets.skew(),
    "Kurtosis": all_synthetic_targets.kurtosis()
})

all_targets_stats = all_targets_stats_raw.copy()

percentage_cols = [
    "Ann. Return",
    "Ann. Volatility",
    "Max Drawdown",
    "Weekly VaR 95%",
    "Weekly Expected Shortfall 95%"
]

for col in percentage_cols:
    all_targets_stats[col] = all_targets_stats[col].map(lambda x: f"{x * 100:.2f}%")

for col in ["Sharpe Ratio", "Skewness", "Kurtosis"]:
    all_targets_stats[col] = all_targets_stats[col].map(lambda x: f"{x:.2f}")

print("Comparison of synthetic targets:")
display(all_targets_stats)

In [ ]:
# Plot cumulative performance of all synthetic targets
all_targets_cumulative = (1 + all_synthetic_targets).cumprod()

plt.figure(figsize=(14, 7))

for col in all_targets_cumulative.columns:
    plt.plot(
        all_targets_cumulative.index,
        all_targets_cumulative[col],
        linewidth=2,
        label=col
    )

plt.title("Cumulative Performance of Synthetic Targets")
plt.xlabel("Date")
plt.ylabel("Cumulative growth of 1 unit invested")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Preliminary replicability check: correlations with futures
replicability_rows = []

for target_name in all_synthetic_targets.columns:
    target_series = all_synthetic_targets[target_name]
    
    corr_with_futures = futures_returns.corrwith(target_series).sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
    
    top_future = corr_with_futures.index[0]
    
    replicability_rows.append({
        "Target": target_name,
        "Top Future": top_future,
        "Top Future Description": variable_info.get(top_future, top_future),
        "Top Correlation": corr_with_futures.iloc[0],
        "Average Top 3 Abs Corr": corr_with_futures.abs().head(3).mean()
    })

replicability_summary = pd.DataFrame(replicability_rows)

replicability_summary["Top Correlation"] = replicability_summary["Top Correlation"].map(
    lambda x: f"{x:.3f}"
)
replicability_summary["Average Top 3 Abs Corr"] = replicability_summary[
    "Average Top 3 Abs Corr"
].map(lambda x: f"{x:.3f}")

print("Preliminary replicability summary:")
display(replicability_summary)

## 4.1 Autocorrelation check across different synthetic targets

In [ ]:
# Ljung-Box test across all synthetic targets
synthetic_targets_ljung_box = []

for target_name in all_synthetic_targets.columns:
    series = all_synthetic_targets[target_name].dropna()
    
    lb_test = acorr_ljungbox(
        series,
        lags=[5, 10, 15, 20],
        return_df=True
    )
    
    for lag in [5, 10, 15, 20]:
        synthetic_targets_ljung_box.append({
            "Target": target_name,
            "Lag": lag,
            "Ljung-Box statistic": lb_test.loc[lag, "lb_stat"],
            "p-value": lb_test.loc[lag, "lb_pvalue"]
        })

synthetic_targets_ljung_box = pd.DataFrame(synthetic_targets_ljung_box)

synthetic_targets_ljung_box_display = synthetic_targets_ljung_box.copy()

synthetic_targets_ljung_box_display["Ljung-Box statistic"] = synthetic_targets_ljung_box_display[
    "Ljung-Box statistic"
].map(lambda x: f"{x:.2f}")

synthetic_targets_ljung_box_display["p-value"] = synthetic_targets_ljung_box_display[
    "p-value"
].map(lambda x: f"{x:.4f}")

print("Ljung-Box test across synthetic targets:")
display(synthetic_targets_ljung_box_display)

## ARCH test across all synthetic targets


In [ ]:
arch_targets_results = []

for target_name in all_synthetic_targets.columns:
    r = all_synthetic_targets[target_name].dropna().values
    
    lm_stat, lm_pval, _, _ = het_arch(r, nlags=5)
    
    arch_targets_results.append({
        "Target": target_name,
        "ARCH LM Statistic": lm_stat,
        "p-value": lm_pval,
        "Volatility Clustering": "Present" if lm_pval < 0.05 else "Not significant"
    })

arch_targets_results = pd.DataFrame(arch_targets_results)

arch_targets_display = arch_targets_results.copy()

arch_targets_display["ARCH LM Statistic"] = arch_targets_display[
    "ARCH LM Statistic"
].map(lambda x: f"{x:.2f}")

arch_targets_display["p-value"] = arch_targets_display[
    "p-value"
].map(lambda x: f"{x:.2e}")

print("ARCH test across synthetic targets:")
display(arch_targets_display)

## 6.8 PCA on futures returns

In [ ]:
# PCA on futures returns
pca = PCA()
pca.fit(futures_returns.dropna())

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.bar(range(1, len(explained) + 1), explained * 100, alpha=0.8)
ax1.plot(range(1, len(explained) + 1), explained * 100, "o-", linewidth=1.5)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained variance (%)")
ax1.set_title("Scree Plot - PCA on Futures Returns")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(cumulative) + 1), cumulative * 100, "o-", linewidth=2)
ax2.axhline(80, linestyle="--", alpha=0.7, label="80% threshold")
ax2.axhline(90, linestyle="--", alpha=0.7, label="90% threshold")
ax2.set_xlabel("Number of components")
ax2.set_ylabel("Cumulative explained variance (%)")
ax2.set_title("Cumulative Explained Variance - Futures PCA")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=[variable_info.get(c, c) for c in futures_returns.columns],
    columns=[f"PC{i+1}" for i in range(5)]
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    loadings,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5
)

plt.title("PCA Loadings - First 5 Principal Components")
plt.tight_layout()
plt.show()

n_80 = np.searchsorted(cumulative, 0.80) + 1
n_90 = np.searchsorted(cumulative, 0.90) + 1

print(f"PCs needed to explain >= 80% of variance: {n_80}")
print(f"PCs needed to explain >= 90% of variance: {n_90}")
print(f"Variance explained by first 3 PCs      : {cumulative[2] * 100:.1f}%")

## 6.9 Deliverable summary

In [ ]:
# Deliverable summary

print("=" * 60)
print("DELIVERABLE SUMMARY")
print("=" * 60)

print("target_returns_aligned")
print(f"  Type   : {type(target_returns_aligned).__name__}")
print(f"  Shape  : {target_returns_aligned.shape}")
print(f"  Name   : '{target_returns_aligned.name}'")
print(f"  Range  : {target_returns_aligned.index[0].date()} → {target_returns_aligned.index[-1].date()}")

print()

print("futures_returns")
print(f"  Type   : {type(futures_returns).__name__}")
print(f"  Shape  : {futures_returns.shape}")
print(f"  Columns: {futures_returns.columns.tolist()}")
print(f"  Range  : {futures_returns.index[0].date()} → {futures_returns.index[-1].date()}")

print()

print(f"Indices aligned: {(target_returns_aligned.index == futures_returns.index).all()}")
print("=" * 60)

## 7.5 Ridge and Lasso Regression using Optuna


In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import MinMaxScaler

def run_rolling_regularized(model_type, alpha, rolling_window):
    # Initialize arrays to store results
    weights_history = []  
    replica_returns = []  
    target_dates = []  
    gross_exposures = []  
    var_values = []  
    scaling_factors = []  

    # Assuming X_values, y_values, and dates_array are pre-defined numpy arrays from your DataFrame
    X_values = X.values
    y_values = y.values
    dates_array = X.index.values

    for i in range(len(X) - rolling_window - 1):
        start_idx = i
        end_idx = i + rolling_window

        # Extract training data
        X_train = X_values[start_idx:end_idx]
        y_train = y_values[start_idx:end_idx]

        # Regularized models REQUIRE normalization
        scaler_X = MinMaxScaler()
        X_train_normalized = scaler_X.fit_transform(X_train)

        scaler_y = MinMaxScaler()
        y_train_normalized = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()

        # Select and fit the requested model
        if model_type == 'Lasso':
            model = Lasso(alpha=alpha, fit_intercept=False, max_iter=10000, tol=1e-4)
        elif model_type == 'Ridge':
            model = Ridge(alpha=alpha, fit_intercept=False, max_iter=10000, tol=1e-4)
        else:
            raise ValueError("model_type must be either 'Lasso' or 'Ridge'")

        model.fit(X_train_normalized, y_train_normalized)

        # Get the normalized weights and scale them back to original scale
        normalized_weights = model.coef_
        original_weights = normalized_weights / scaler_X.scale_

        scaling_factor = 1.0

        # VaR Risk Scaling Block 
        if len(replica_returns) >= 12:  
            historical_returns = []
            for j in range(max(0, len(replica_returns)-52), len(replica_returns)):
                hist_returns = X_values[end_idx-(len(replica_returns)-j)]
                weighted_return = np.dot(hist_returns, original_weights)
                historical_returns.append(weighted_return)

            # Assuming calculate_var is a pre-defined custom function
            var = calculate_var(
                historical_returns,
                confidence=0.01,
                horizon=4
            )

            if var > max_var_threshold:
                scaling_factor = max_var_threshold / var
                original_weights = original_weights * scaling_factor

                scaled_historical_returns = [ret * scaling_factor for ret in historical_returns]
                scaled_var = calculate_var(
                    scaled_historical_returns,
                    confidence=0.01,
                    horizon=4
                )
                var = scaled_var  

            var_values.append(var)
        else:
            var_values.append(np.nan)

        scaling_factors.append(scaling_factor)

        gross_exposure = np.sum(np.abs(original_weights))
        gross_exposures.append(gross_exposure)
        weights_history.append(original_weights)

        # Calculate out-of-sample return for t+1
        next_returns = X_values[end_idx]  
        replica_return = np.dot(next_returns, original_weights)
        replica_returns.append(replica_return)
        target_dates.append(dates_array[end_idx])

    replica_returns_series = pd.Series(replica_returns, index=target_dates)

    # Calculate cumulative returns
    aligned_target = y.loc[replica_returns_series.index]
    cumulative_target = (1 + aligned_target).cumprod()
    cumulative_replica = (1 + replica_returns_series).cumprod()

    # Calculate performance metrics (Assuming weekly data based on *52)
    replica_mean_return = replica_returns_series.mean() * 52  
    target_mean_return = aligned_target.mean() * 52  

    replica_vol = replica_returns_series.std() * np.sqrt(52)  
    target_vol = aligned_target.std() * np.sqrt(52)  

    replica_sharpe = replica_mean_return / replica_vol if replica_vol > 0 else 0
    target_sharpe = target_mean_return / target_vol if target_vol > 0 else 0

    tracking_error = (replica_returns_series - aligned_target).std() * np.sqrt(52)
    information_ratio = (replica_mean_return - target_mean_return) / tracking_error if tracking_error > 0 else 0

    replica_drawdown = 1 - cumulative_replica / cumulative_replica.cummax()
    target_drawdown = 1 - cumulative_target / cumulative_target.cummax()

    correlation = replica_returns_series.corr(aligned_target)
    avg_gross_exposure = np.mean(gross_exposures)
    avg_var = np.nanmean(var_values)

    return {
        'model_type': model_type,
        'alpha': alpha,
        'rolling_window': rolling_window,
        'replica_return': replica_mean_return,
        'target_return': target_mean_return,
        'replica_vol': replica_vol,
        'target_vol': target_vol,
        'replica_sharpe': replica_sharpe,
        'target_sharpe': target_sharpe,
        'tracking_error': tracking_error,
        'information_ratio': information_ratio,
        'correlation': correlation,
        'max_drawdown': replica_drawdown.max(),
        'avg_gross_exposure': avg_gross_exposure,
        'avg_var': avg_var,
        'replica_returns': replica_returns_series,
        'aligned_target': aligned_target,
        'cumulative_replica': cumulative_replica,
        'cumulative_target': cumulative_target,
        'gross_exposures': gross_exposures,
        'var_values': var_values,
        'scaling_factors': scaling_factors,
        'weights_history': weights_history
    }

In [ ]:
# GRID SEARCH CELL
model_types = ['Lasso', 'Ridge']
alphas = [0.0001, 0.001, 0.01, 0.1] 
rolling_windows = [52, 104, 156]  # in weeks (1Y, 2Y, 3Y)

results = []
for model_type in model_types:
    for window in rolling_windows:
        for alpha in alphas:
            result = run_rolling_regularized(model_type, alpha, window)
            results.append(result)
            print(f"Completed: {model_type} | Window: {window} weeks | Alpha: {alpha}")

results_df = pd.DataFrame(results)

# Sort by tracking error (lower is better)
sorted_results = results_df.sort_values(by='tracking_error', ascending=True)

# Display top configurations
print("\nTop configurations by Tracking error:")
display(sorted_results[['model_type', 'alpha', 'rolling_window', 'information_ratio', 
                        'correlation', 'tracking_error', 'replica_sharpe', 
                        'avg_gross_exposure', 'avg_var']].head(10))

# Get the best configuration
best_config = sorted_results.iloc[0]
print(f"\nBest configuration: {best_config['model_type']} | Alpha={best_config['alpha']} | Window={best_config['rolling_window']} weeks")
print(f"Tracking error: {best_config['tracking_error']:.4f}")

In [ ]:
# PLOT FOR THE BEST CONFIGURATION


# Create detailed metrics table
metrics = pd.DataFrame({
    'Metric': ['Annualized return', 'Annualized volatility', 'Sharpe ratio',
               'Max Drawdown', 'Tracking Error', 'Information ratio',
               'Correlation', 'Average gross exposure', 'Average VaR (1%, 1M)'],
    'Target': [f"{best_config['target_return']*100:.2f}%",
               f"{best_config['target_vol']*100:.2f}%",
               f"{best_config['target_sharpe']:.2f}",
               f"{best_config['max_drawdown']*100:.2f}%",
               "N/A", "N/A", "N/A", "N/A", "N/A"],
    'Replica': [f"{best_config['replica_return']*100:.2f}%",
                f"{best_config['replica_vol']*100:.2f}%",
                f"{best_config['replica_sharpe']:.2f}",
                f"{best_config['max_drawdown']*100:.2f}%",
                f"{best_config['tracking_error']*100:.2f}%",
                f"{best_config['information_ratio']:.2f}",
                f"{best_config['correlation']:.4f}",
                f"{best_config['avg_gross_exposure']:.4f}",
                f"{best_config['avg_var']*100:.2f}%"]
})

print(f"\nDetailed metrics for the best configuration ({best_config['model_type']}):")
display(metrics)

# Plot cumulative returns
plt.figure(figsize=(12, 6))
plt.plot(best_config['cumulative_target'], label='Target index', color='blue')
plt.plot(best_config['cumulative_replica'], label='Replica portfolio', color='red')
plt.title(f"Cumulative returns: target vs replica ({best_config['model_type']})")
plt.xlabel('Date')
plt.ylabel('Cumulative return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot drawdowns
plt.figure(figsize=(12, 6))
target_dd_plot = 1 - best_config['cumulative_target'] / best_config['cumulative_target'].cummax()
replica_dd_plot = 1 - best_config['cumulative_replica'] / best_config['cumulative_replica'].cummax()
plt.plot(target_dd_plot, label='Target index', color='blue')
plt.plot(replica_dd_plot, label='Replica portfolio', color='red')
plt.title(f"Drawdowns: target vs replica ({best_config['model_type']})")
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot VaR over time
plt.figure(figsize=(12, 6))
plt.plot(best_config['var_values'], color='orange')
plt.axhline(y=max_var_threshold, color='r', linestyle='--', label=f'VaR threshold ({max_var_threshold*100}%)')
plt.title(f"Value at Risk (VaR) over time ({best_config['model_type']})")
plt.xlabel('Date')
plt.ylabel('VaR (1%, 1M)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
